<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/t5_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate evaluate

In [ ]:
import re
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

# NQ dataset

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train")
eval_dataset = nq.select(range(1000))
dataset_name = "natural-questions"

# Helper functions

In [ ]:
def get_question(example):
  return example["query"]

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0] if len(answer) > 0 else ""
  return str(answer).strip()

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# F1 Metrics

In [ ]:
def qa_f1(prediction, gold):
    pred_tokens = normalize_text(prediction).split()
    gold_tokens = normalize_text(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return int(pred_tokens == gold_tokens)

    common = set(pred_tokens) & set(gold_tokens)
    num_same = sum(
        min(pred_tokens.count(tok), gold_tokens.count(tok))
        for tok in common
    )

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

# Load T5

In [ ]:
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

model.eval()

print("Loaded model: ", model_name)

# Converting nq into T5 input format

In [ ]:
def make_input(question):
    return f"question: {question}"

def generate_t5_answer(question, max_new_tokens=32):
    input_text = make_input(question)

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4
        )

    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer

# Evaluate

In [ ]:
results = []

for i in range(len(eval_dataset)):
    example = eval_dataset[i]

    question = get_question(example)
    gold = get_answer(example)
    pred = generate_t5_answer(question, max_new_tokens=32)

    f1 = qa_f1(pred, gold)

    results.append({
        "dataset": dataset_name,
        "question_number": i,
        "question": question,
        "gold_answer": gold,
        "prediction": pred,
        "f1": f1
    })

results_df = pd.DataFrame(results)
results_df.to_csv("t5_eval_results.csv", index=False)

print("Saved t5_eval_results.csv")
print("Final T5 F1:", results_df["f1"].mean() * 100)

In [ ]:
for i in range(5):
    print("QUESTION:", results_df["question"][i])
    print("GOLD:", results_df["gold_answer"][i])
    print("PRED:", results_df["prediction"][i])
    print("F1:", results_df["f1"][i])
    print("-" * 80)